Import Necessary packages
---

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

os.chdir('/home/cyf/wbi/Virginia/wbi_0811/wholistic_registration')
print(os.getcwd())

import h5py
from utils import preprocess, calFlow3d_Wei_v1, visualization,mask,option,IO,reference
from utils import registration
import numpy as np
import nd2
import tifffile as tiff
from matplotlib import pyplot as plt
import cupy as cp

#filePath
movingFilePath="/home/cyf/wbi/Virginia/f338/221124_f338_ubi_gCaMP7f_bactin_mCherry_CAAX_7dpf002.nd2"
config_file='/home/cyf/wbi/Virginia/wbi_0811/wholistic_registration/config_338.toml'
base_folder="/home/cyf/wbi/Virginia/f338_1023registraed/"
# os.makedirs(base_folder, exist_ok=True)

mem_folder = os.path.join(base_folder, "membrane")
ca_folder = os.path.join(base_folder, "ca")
concat_folder= os.path.join(base_folder, "concat")

os.makedirs(mem_folder, exist_ok=True)
os.makedirs(ca_folder, exist_ok=True)
os.makedirs(concat_folder, exist_ok=True)
#read meta data
meta_moving=IO.readMeta(movingFilePath)

#process 2000 frames in 1 iteration
total_frames = 1598
chunk_size = 200
def compute_reference_from_block(mem_block, ca_block,frames=20):
    """Generate a reference image from a block of frames"""
    mem_block = cp.asarray(mem_block)
    mem_ref, indsort = reference.pick_initial_reference(mem_block,max_corr_frames=20)
    ca_block = cp.asarray(ca_block)
    Ca_ref = cp.mean(ca_block[indsort, :], axis=0)
    Ca_ref_transform = 50 * cp.log10(1 + Ca_ref)

    return (mem_ref + Ca_ref_transform).get()


def register_one_frame(frame_idx, mem_img, ca_img, ref_pool, ref_range,verbose=True):
    """Register one mem+Ca frame to the reference generated from the pool"""
    ref_img = compute_reference_from_block(
        np.array([m for m in ref_pool["mem"]]),
        np.array([c for c in ref_pool["ca"]])
    )
    mem_reg, ca_reg, _, _, motion_reg = registration.wbi_registration_3d(
        np.expand_dims(mem_img, axis=0),
        np.expand_dims(ca_img, axis=0),
        config_file,
        ref_img,
        verbose=verbose
    )
    return mem_reg[0], ca_reg[0], ref_img, motion_reg[0]

/home/cyf/wbi/Virginia/wbi_0811/wholistic_registration
CuPy is available with CUDA - using GPU acceleration
Z ratio is 27.69230769230769
Data size is (2304, 1708, 13)
Total frames is 1598


In [3]:
import os
import numpy as np
import zarr
from numcodecs import GZip

# ======================
# Global variables
# ======================
references_log = []  # store reference sources
motionFilePath="/home/cyf/wbi/Virginia/f338_1107motion.zarr"
# ======================
# Utility functions
# ======================



# ======================
# Main pipeline
# ======================

# Read one frame to get image size
test_img = IO.readFrame(movingFilePath, 0, channel=0)
H, W, D = test_img.shape
# Downsample: take 1 frame every 10 frames
downsample = 1
num_frames_ds = total_frames // downsample

from numcodecs import Blosc
compressor = Blosc(cname="lz4", clevel=1, shuffle=Blosc.BITSHUFFLE)

# Pre-create a large Zarr file (store downsampled frames + reference)
root_motion=zarr.open(motionFilePath,mode='w')



motion_z = root_motion.create_dataset(
    "motion", shape=(num_frames_ds, 4, W, H, 3),
    chunks=(1, 4, W, H, 3), dtype="f4", compressor=None
)
ref_z = root_motion.create_dataset(
    "reference", shape=(num_frames_ds, 4, W, H),
    chunks=(1, 4, W, H), dtype="f4", compressor=None
)



# Save config
with open(config_file, "r") as f:
    root_motion.attrs["config"] = f.read()


# ==== Step 1: process the middle block ====
chunk_size = 200
half_chunk = chunk_size // 2
total_mid = total_frames // 2
mid_start = total_mid - half_chunk
mid_end   = mid_start + chunk_size

frames_mid = list(range(mid_start, mid_end, downsample))
# print(frames_mid)

mem_mid = IO.readFrame(movingFilePath, frames_mid,[4,5,6,7] ,channel=1)
ca_mid  = IO.readFrame(movingFilePath, frames_mid,[4,5,6,7], channel=0)
print("Finish loading the data of the middle block")
# reference = generated from the middle block itself

ref_img = compute_reference_from_block(mem_mid, ca_mid,frames=35)
print("Finish picking the initial reference from the middle block")

mem_mid, ca_mid, _, _, motion_mid = registration.wbi_registration_3d(
    mem_mid, ca_mid, config_file, ref_img
)
print(motion_mid[0].shape)
print(mem_mid[0].shape)
print("Finish processing the middle block")
start_idx = mid_start // downsample
end_idx   = mid_end   // downsample

batch_size = 10  # 例如每次写10帧
for i in range(0, len(motion_mid), batch_size):
    j = min(i + batch_size, len(motion_mid))
    print(f"Writing frames {i}-{j-1}")
    motion_batch=cp.asnumpy(motion_mid[i:j]).astype("f4")

    ref_batch = np.repeat(ref_img[None, :, :], j - i, axis=0).astype("f4")
    ref_z[start_idx + i:start_idx + j] = ref_batch.transpose(0,3,2,1)
    motion_z[start_idx + i:start_idx + j]=motion_batch.transpose(0,3,2,1,4)
#motion[start_idx:end_idx]= np.asarray(motion_mid).astype("f4")
references_log.append({
    "used_for": (mid_start, mid_end),
    "ref_from": (mid_start, mid_end),
    "ref_img": ref_img
})
print(f"Middle block {mid_start}~{mid_end-1} finished and set as anchor reference")

Finish loading the data of the middle block
Finish picking the initial reference from the middle block
Frame: 1	Initial Error is:439.3752746582031	Eventual Error: 147.7590789794922
Frame: 2	Initial Error is:486.8858947753906	Eventual Error: 146.9805145263672
Frame: 3	Initial Error is:495.652099609375	Eventual Error: 142.5876922607422
Absolute difference between old and new error is less than 1e-3
Frame: 4	Initial Error is:442.02239990234375	Eventual Error: 142.97271728515625
Frame: 5	Initial Error is:431.7760314941406	Eventual Error: 147.22669982910156
Frame: 6	Initial Error is:472.7721862792969	Eventual Error: 157.5088348388672
Frame: 7	Initial Error is:395.09515380859375	Eventual Error: 144.07192993164062
Frame: 8	Initial Error is:402.0565185546875	Eventual Error: 145.6488800048828
Frame: 9	Initial Error is:396.45458984375	Eventual Error: 142.5697021484375
Frame: 10	Initial Error is:408.3499755859375	Eventual Error: 145.00941467285156
Frame: 11	Initial Error is:397.9296875	Eventual E

In [ ]:
# ==== Step 2: process backward (downsampled) ====
window_size = 50

ref_windows_mem=np.array(mem_mid[0:50])
ref_windows_ca=np.array(ca_mid[0:50])
print(ref_windows_mem.shape)

for idx in range(mid_start - downsample, -1, -downsample):
    ref_start = (idx + downsample) // downsample
    ref_end   = min((idx + window_size * downsample) // downsample, num_frames_ds)

    mem_ref_block = ref_windows_mem
    ca_ref_block  = ref_windows_ca

    mem_img = IO.readFrame(movingFilePath, idx, [4,5,6,7],channel=1)
    ca_img  = IO.readFrame(movingFilePath, idx, [4,5,6,7],channel=0)

    mem_reg, ca_reg, ref_img,motion_reg = register_one_frame(idx, mem_img, ca_img,
                                                  {"mem": mem_ref_block, "ca": ca_ref_block},
                                                  (ref_start * downsample, ref_end * downsample),verbose=False)

    z_idx = idx // downsample
    ref_z[z_idx] = ref_img.astype("f4").transpose(2,1,0)
    motion_z[z_idx]=motion_reg.transpose(2,1,0,3)

    #updata the window
    mem_ref_block[1:window_size]=mem_ref_block[0:window_size-1]
    mem_ref_block[0]=mem_reg
    ca_ref_block[1:window_size]=ca_ref_block[0:window_size-1]
    ca_ref_block[0]=ca_reg
    #motion[z_idx] = motion_reg.astype("f4")
    print(f"Processed backward up to frame {idx}")


(50, 1708, 2304, 4)
Frame: 1	Initial Error is:414.95672607421875	Eventual Error: 114.73477172851562
Processed backward up to frame 698
Frame: 1	Initial Error is:527.8153686523438	Eventual Error: 121.3601303100586
Processed backward up to frame 697
Frame: 1	Initial Error is:451.7830505371094	Eventual Error: 113.1863784790039
Processed backward up to frame 696
Frame: 1	Initial Error is:462.4847412109375	Eventual Error: 107.33219146728516
Processed backward up to frame 695
Frame: 1	Initial Error is:412.2361755371094	Eventual Error: 106.24821472167969
Processed backward up to frame 694
Frame: 1	Initial Error is:410.9219970703125	Eventual Error: 104.89893341064453
Processed backward up to frame 693
Frame: 1	Initial Error is:403.99700927734375	Eventual Error: 110.06277465820312
Processed backward up to frame 692
Frame: 1	Initial Error is:472.6329650878906	Eventual Error: 106.87675476074219
Processed backward up to frame 691
Frame: 1	Initial Error is:513.0759887695312	Eventual Error: 104.3260

In [16]:
# ==== Step 3: process forward (downsampled) ====
window_size = 50
ref_windows_mem=np.array(mem_mid[-50:])
ref_windows_ca=np.array(ca_mid[-50:])
print(ref_windows_mem.shape)
for idx in range(mid_end, total_frames, downsample):
    ref_start = max(0, (idx - window_size * downsample) // downsample)
    ref_end   = idx // downsample

    mem_ref_block = ref_windows_mem
    ca_ref_block  = ref_windows_ca

    mem_img = IO.readFrame(movingFilePath, idx, [4,5,6,7],channel=1)
    ca_img  = IO.readFrame(movingFilePath, idx, [4,5,6,7],channel=0)

    mem_reg, ca_reg, ref_img,motion_reg = register_one_frame(idx, mem_img, ca_img,
                                                  {"mem": mem_ref_block, "ca": ca_ref_block},
                                                  (ref_start * downsample, ref_end * downsample),verbose=False)
                
    ref_z[z_idx] = ref_img.astype("f4").transpose(2,1,0)
    motion_z[z_idx]=motion_reg.transpose(2,1,0,3)

    #updata the window
    mem_ref_block[0:window_size-1]=mem_ref_block[1:window_size]
    mem_ref_block[-1]=mem_reg
    ca_ref_block[0:window_size-1]=ca_ref_block[1:window_size]
    ca_ref_block[-1]=ca_reg
    print(f"Processed forward up to frame {idx}")



(50, 1708, 2304, 4)
Frame: 1	Initial Error is:479.802734375	Eventual Error: 98.95572662353516
Processed forward up to frame 899
Frame: 1	Initial Error is:569.3214721679688	Eventual Error: 98.92828369140625
Processed forward up to frame 900
Frame: 1	Initial Error is:520.528076171875	Eventual Error: 100.79359436035156
Processed forward up to frame 901
Frame: 1	Initial Error is:570.892822265625	Eventual Error: 98.69951629638672
Processed forward up to frame 902
Frame: 1	Initial Error is:525.0459594726562	Eventual Error: 97.2507553100586
Processed forward up to frame 903
Frame: 1	Initial Error is:500.3516845703125	Eventual Error: 97.71037292480469
Processed forward up to frame 904
Frame: 1	Initial Error is:546.7493286132812	Eventual Error: 99.8443374633789
Processed forward up to frame 905
Frame: 1	Initial Error is:551.9076538085938	Eventual Error: 96.38571166992188
Processed forward up to frame 906
Frame: 1	Initial Error is:538.148681640625	Eventual Error: 95.4498291015625
Processed forwa